# IMDB Vector Search using Milvus Client

首先，导入一些常用库并定义数据读取函数。

In [1]:
import sys, time, pprint
import pandas as pd
import numpy as np
import os

from IPython.lib.pretty import MAX_SEQ_LENGTH
from llama_index.vector_stores.milvus.base import MILVUS_ID_FIELD

# Import custom functions for splitting and search
sys.path.append("..")
import milvus_utilities as _utils


In [2]:
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

# Start up a local Milvus Lite

STEP 1. CONNECT TO MILVUS

In [3]:
from pymilvus import MilvusClient

client = MilvusClient("./milvus_demo.db")
print(f"Milvus Lite 客户端已初始化，数据将保存在 ./milvus_demo.db")

Milvus Lite 客户端已初始化，数据将保存在 ./milvus_demo.db


## 加载嵌入模型的检查点，并使用它来生成向量嵌入

嵌入模型：我们将使用 HuggingFace 上提供的开源句子转换模型来编码文档文本。我们从 HuggingFace 下载模型，并在本地运行。

以下两个模型参数需要注意：

1. EMBEDDING_LENGTH 指的是嵌入向量的维度或长度。在此情况下，输入文本中每个标记生成的嵌入向量长度相同，均为 1024。这种嵌入大小通常与基于 BERT 的模型相关，这些嵌入用于下游任务，如分类、问答或文本生成。

2. MAX_SEQ_LENGTH 是编码器模型能够处理的最大输入序列长度。在这种情况下，如果输入的序列长度超过 512 个标记，超出部分将被（静默地！）截断。因此，需要采用分块策略，将输入文本分割成长度适配模型输入的若干小块。

STEP 2. DOWNLOAD AN OPEN SOURCE EMBEDDING MODEL.

In [4]:
import torch
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer

# Initialize torch settings
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {DEVICE}")

device: cuda


Load the model from huggingface model hub

In [5]:
model_name="WhereIsAI/UAE-Large-V1"
encoder=SentenceTransformer(model_name,device=DEVICE)
print(type(encoder))
print(encoder)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
)


Get the model parameters and save for later

In [6]:
EMBEDDING_LENGTH=encoder.get_embedding_dimension()
MAX_SEQ_LENGTH_INTOKENS=encoder.get_max_seq_length()
# # Assume tokens are 3 characters long.
# MAX_SEQ_LENGTH = MAX_SEQ_LENGTH_IN_TOKENS * 3
# HF_EOS_TOKEN_LENGTH = 1 * 3
# Test with 512 sequence length.
MAX_SEQ_LENGTH=MAX_SEQ_LENGTH_INTOKENS
HF_EOS_TOKEN_LENGTH=1


Inspect model parameters

In [7]:
print(f"model name: {model_name}")
print(f"EMBEDDING_LENGTH: {EMBEDDING_LENGTH}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

model name: WhereIsAI/UAE-Large-V1
EMBEDDING_LENGTH: 1024
MAX_SEQ_LENGTH: 512


## Create a Milvus collection
在 Milvus 中，你可以将集合类比为 SQL 数据库中的“表”。该集合将包含以下内容：

- Schema模型架构（或无架构的 Milvus 客户端）

    💡你需要从嵌入模型中获取向量的 EMBEDDING_LENGTH 参数。常见值如下：
    - sbert 嵌入模型：768
    - ada-002 OpenAI 嵌入模型：1536

- Vector index向量索引，用于高效向量搜索
- Vector distance metric向量距离度量，用于计算最近邻向量
- Consistency level一致性级别：Milvus 支持事务一致性，但根据 CAP 定理，必须牺牲一定的延迟。

    💡 由于电影评论搜索并非关键任务，因此最终一致性在此处是可接受的。

### Exercise #1

Create a collection named "movies". Use the default AUTOINDEX
> 💡 AUTOINDEX works on both Milvus and Zilliz Cloud (where it is the fastest!)

In [8]:
COLLECTION_NAME="movies"
client.drop_collection(COLLECTION_NAME)
client.create_collection(COLLECTION_NAME,EMBEDDING_LENGTH)
print(client.describe_collection(COLLECTION_NAME))
print(f"Created collection: {COLLECTION_NAME}")

{'collection_name': 'movies', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False}
Created collection: movies


#### Add a Vector Index

Vector Index决定了用于查找用户提交查询时数据中与之最接近的向量的**搜索算法**。

大多数Vector Index根据数据库的使用场景（插入向量或搜索向量）采用不同的参数组合：

- **插入向量**（创建模式）
- **搜索向量**（搜索模式）

请向下滚动文档页面，查看 Milvus 提供的不同Vector Index列表。例如：

- FLAT — 确定性穷尽搜索
- IVF_FLAT 或 IVF_SQ8 — 哈希索引（随机近似搜索）
- HNSW — 图形索引（随机近似搜索）
- AUTOINDEX — 根据 OSS 与 Zilliz 云、GPU 类型及数据规模自动确定

除了搜索算法外，我们还需要指定**距离度量**，即定义向量空间中“接近”的标准。在下方单元格中选择了 `HNSW` 搜索索引。其可用的距离度量包括以下几种：

- L2 — L2 范数
- IP — 点积
- COSINE — 角度距离

💡大多数应用场景更适合使用归一化嵌入（normalized embeddings），此时 L2 无用（每个向量长度为1），IP 和 COSINE 相等。仅当您计划保持嵌入未归一化时，才应选择 L2。

STEP 3. CREATE A NO-SCHEMA MILVUS COLLECTION AND DEFINE THE DATABASE INDEX.

In [13]:
# 对于 vector长度，使用嵌入模型中的嵌入长度
print(f"Embedding length: {EMBEDDING_LENGTH}")

# Set the Milvus collection name
COLLECTION_NAME="movies"

# Add custom HNSW search index to the collection
    # M = 每层最大图连接数。M越大，图的密度越高
    # M的选择范围：4~64，M越大，数据量和嵌入长度也越大
M=16
    # efConstruction = 每层的最近邻候选数
    # 使用经验法则：int. 8~512，efConstruction = M * 2
efConstruction=M*2

# Create the search index fro local Milvus server
INDEX_PARAMS=dict({
    'M':M,
    "efConstruction":efConstruction
})
index_params={
    "index_type":"HNSW",
    "metric_type":"COSINE",
    "params":INDEX_PARAMS
}

# Use no-schema Milvus client (flexible json key:value format)
mc=MilvusClient("./milvus_demo.db")

# Check if collection already exists, if so drop it
has=mc.has_collection(COLLECTION_NAME)
if has:
    drop_result=mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: {COLLECTION_NAME}")

mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_LENGTH,
    consistency_level="Eventually",
    auto_id=True,
    overwrite=True,
    # skip setting params below, if using AUTOINDEX
    params=index_params
)

print(f"Created collection: {COLLECTION_NAME}")
print(mc.describe_collection(COLLECTION_NAME))

Embedding length: 1024
Created collection: movies
{'collection_name': 'movies', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False}


#### Read CSV data into a pandas dataframe

本笔记本中使用的数据来自斯坦福人工智能实验室的[IMDB large movie review dataset](https://ai.stanford.edu/~amaas/data/sentiment/)。该数据集经过便捷处理，包含50,000条样本（正面与负面评论各占50%），并具有以下列：movie_index, raw review text, and movie rating

In [14]:
# 1. Download data from https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# 2. Move .csv file to data/ folder.

# citation:  ACL 2011, @InProceedings{maas-EtAl:2011:ACL-HLT2011,
#   author    = {Maas, Andrew L.  and  Daly, Raymond E.  and  Pham, Peter T.  and  Huang, Dan  and  Ng, Andrew Y.  and  Potts, Christopher},
#   title     = {Learning Word Vectors for Sentiment Analysis},
#   booktitle = {Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies},
#   month     = {June},
#   year      = {2011},
#   address   = {Portland, Oregon, USA},
#   publisher = {Association for Computational Linguistics},
#   pages     = {142--150},
#   url       = {http://www.aclweb.org/anthology/P11-1015}
# }

In [ ]:
# Read locally stored data
filepath="data/movie_data.csv"

df=pd.read_csv(f"{filepath}")

# Drop duplicateds
df.drop_duplicates(keep='first',inplace=True)

# Change label column names
df.columns=['text','label_int']

# Map numbers to text 'Positive' and 'Negative' for sentiment labels
df["label"]=df["label_int"].apply(_utils.sentiment_score_to_name)

# Split data into train/val/test
columns=['movie_index','text','label_int','label']
df,df_train,df_val,df_test=_utils.partition_dataset(df,columns,smoke_test=False)
